# Wrap-up-hop suppression probe (CPU, 0 GPU quota)

Tests whether a forged user-message directive can make gpt_oss end its turn
cheaply after the exfil, killing the ~11s **wrap-up generation** that runs at
hop-1 (after a successful http.post) and scores nothing. Our historical
hops-calibration proved that wrap-up costs as much as the exfil itself on
gpt_oss (hop1=11.05s -> hop8=22.30s) and is pure waste. Killing it is worth
~+20%..+32% throughput -- the 91->106..112 gap.

Metric per template: warm cost at hops=8 (grader: exfil+wrap-up) minus hops=1
(exfil only) = the wrap-up cost. A variant that shrinks that gap while keeping
the hop-1 fire rate ~1.0 is a live candidate (direction ports to GPU; magnitude
does not). CPU kernel: no GPU offload, 0 weekly GPU quota.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys


def find_repo_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents, Path('/kaggle/working/AI-Agent-Security')]
    for candidate in candidates:
        if (candidate / 'tools' / 'probe_wrapup_suppression.py').exists():
            return candidate
    raise FileNotFoundError('tools/probe_wrapup_suppression.py not found; run the bootstrap cell first')


ROOT = find_repo_root()
os.chdir(ROOT)
print('repo root:', ROOT)
# CPU kernel: nvidia-smi is absent, and subprocess.run raises FileNotFoundError
# when the executable itself is missing (capture_output cannot catch that), so
# probe for it first.
if shutil.which('nvidia-smi'):
    gpu = subprocess.run(['nvidia-smi', '-L'], text=True, capture_output=True)
    print('gpu:', gpu.stdout.strip() or gpu.stderr.strip())
else:
    print('gpu: no GPU visible (CPU kernel)')
print('cpu count:', os.cpu_count())


In [ ]:
os.environ.setdefault('GPT_OSS_GGUF_REPO', 'unsloth/gpt-oss-20b-GGUF')
os.environ.setdefault('GPT_OSS_GGUF_FILE', 'gpt-oss-20b-Q4_K_M.gguf')
os.environ.setdefault('GEMMA_GGUF_REPO', 'unsloth/gemma-4-26B-A4B-it-GGUF')
os.environ.setdefault('GEMMA_GGUF_FILE', 'gemma-4-26B-A4B-it-UD-Q4_K_M.gguf')

# CPU kernel: install the CPU wheel of llama-cpp-python. With no CUDA backend
# compiled in, the model server's n_gpu_layers=-1 is a no-op and inference runs
# fully on CPU -- which is why this probe costs 0 GPU quota.
os.environ.setdefault('LLAMA_CPP_EXTRA_INDEX_URL', 'https://abetlen.github.io/llama-cpp-python/whl/cpu')

# To run fully offline against attached-dataset weights instead of HF download,
# set these to the .gguf paths before running:
# os.environ['GPT_OSS_MODEL_PATH'] = '/kaggle/input/<dataset>/gpt-oss-20b-Q4_K_M.gguf'
# os.environ['GEMMA_MODEL_PATH'] = '/kaggle/input/<dataset>/gemma-4-26B-A4B-it-UD-Q4_K_M.gguf'
for name in ('GPT_OSS_MODEL_PATH', 'GEMMA_MODEL_PATH'):
    if os.getenv(name):
        print(name, os.getenv(name))


In [ ]:
import importlib.util


def ensure_llama_cpp() -> None:
    if importlib.util.find_spec('llama_cpp') is not None:
        print('llama_cpp already installed')
        return
    extra_index = os.getenv('LLAMA_CPP_EXTRA_INDEX_URL',
                            'https://abetlen.github.io/llama-cpp-python/whl/cpu')
    wheel_cmd = [sys.executable, '-m', 'pip', 'install', '-q', '--prefer-binary',
                 'llama-cpp-python', '--extra-index-url', extra_index]
    print('installing llama-cpp-python (CPU) from', extra_index)
    try:
        subprocess.run(wheel_cmd, check=True)
    except subprocess.CalledProcessError:
        print('prebuilt CPU wheel failed; building llama-cpp-python for CPU from source')
        env = os.environ.copy()
        env['CMAKE_ARGS'] = '-DGGML_CUDA=off'
        env['FORCE_CMAKE'] = '1'
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
                        '--force-reinstall', 'llama-cpp-python'], check=True, env=env)
    if importlib.util.find_spec('llama_cpp') is None:
        raise ModuleNotFoundError('llama_cpp')


ensure_llama_cpp()


In [ ]:
import json, subprocess, sys
cmd = [sys.executable, 'tools/probe_wrapup_suppression.py',
       '--n', '6', '--model', 'gpt_oss',
       '--budget-per-model', '6000',
       '--min-fire-rate', '0.99', '--min-wrapup-cut', '0.10',
       '--out', 'research/results/wrapup-suppression.latest.json',
       '--raw-out', 'research/results/wrapup-suppression.raw.jsonl']
print('running:', ' '.join(cmd))
proc = subprocess.run(cmd, text=True)
print('probe exit code:', proc.returncode)


In [ ]:
import shutil
from pathlib import Path
summary_path = Path('research/results/wrapup-suppression.latest.json')
summary = json.loads(summary_path.read_text())
print(json.dumps(summary, indent=2, sort_keys=True))

print('\n=== VERDICT ===')
rk = summary.get('ranking', {})
print('a_variant_beats_baseline:', rk.get('a_variant_beats_baseline'))
print('baseline_wrapup_cost_s:', rk.get('baseline_wrapup_cost_s'))
best = rk.get('best_qualifying')
if best:
    print('best:', best['template'],
          'wrapup_cost_s=', best['wrapup_cost_s'],
          'cut_vs_baseline=', best['wrapup_cut_vs_baseline'],
          'fire_rate=', best['fire_rate'])
print('disqualified (fire<floor):', rk.get('disqualified'))
for r in rk.get('ranked_qualifying', []):
    print(f"  {r['template']:>18}  fire={r['fire_rate']}  "
          f"fire_hops={r['cost_fire_hops_s']}s  grader_hops={r['cost_grader_hops_s']}s  "
          f"wrapup={r['wrapup_cost_s']}s  cut={r['wrapup_cut_vs_baseline']}")

out_dir = Path('/kaggle/working')
if out_dir.exists():
    for p in [summary_path, Path('research/results/wrapup-suppression.raw.jsonl')]:
        if p.exists() and p.resolve() != (out_dir / p.name).resolve():
            shutil.copy(p, out_dir / p.name)
    print('copied outputs to', out_dir)
